In [1]:
!pip install transformers pandas gradio streamlit beautifulsoup4 requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 36.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
from transformers import pipeline
import gradio as gr
import re

In [3]:
# Expanded buzzword list (common greenwashing terms)
buzzwords = [
    "eco-friendly", "natural", "green", "sustainable", "planet-safe", "earth-friendly",
    "environmentally conscious", "biodegradable", "organic", "eco-conscious", "responsible",
    "clean", "non-toxic", "chemical-free", "renewable", "ethical", "conscious choice",
    "better for the planet", "low impact", "green choice", "eco option"
]

# Certification keywords (verifiable standards)
certifications = [
    "FSC", "GOTS", "B-Corp", "Fairtrade", "USDA Organic", "Cradle-to-Cradle",
    "ISO 14001", "LEED", "Energy Star", "Rainforest Alliance", "CarbonNeutral",
    "OEKO-TEX", "EPEAT", "Green Seal"
]

In [4]:
def detect_buzzwords(text):
    return [word for word in buzzwords if word.lower() in text.lower()]

def detect_certifications(text):
    return [c for c in certifications if c.lower() in text.lower()]

In [5]:
classifier = pipeline("text-classification", model="roberta-large-mnli")

def classify_claim(text):
    reference = "This product is certified and evidence-based."
    result = classifier(f"{text} </s> {reference}")
    label = result[0]['label']
    if label == "CONTRADICTION":
        return "Marketing Fluff"
    elif label == "ENTAILMENT":
        return "Evidence-Based"
    else:
        return "Unverifiable"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.43G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
def audit_product(text):
    buzz = detect_buzzwords(text)
    certs = detect_certifications(text)
    ml_class = classify_claim(text)

    if certs:
        classification = "Evidence-Based"
        reasoning = f"Mentions certification: {', '.join(certs)}"
    elif buzz:
        classification = "Marketing Fluff"
        reasoning = f"Contains vague buzzwords: {', '.join(buzz)} without certification."
    else:
        classification = ml_class
        reasoning = "Fallback to ML model classification."

    return {
        "Input": text,
        "Buzzwords": buzz,
        "Certifications": certs,
        "Classification": classification,
        "Reasoning": reasoning
    }

In [7]:
print(audit_product("This shirt is eco-friendly and natural."))
print(audit_product("This cotton is GOTS-certified and produced by a B-Corp brand."))
print(audit_product("Our packaging is 100% recyclable and FSC-certified."))
print(audit_product("Made with sustainable materials for a greener future."))

{'Input': 'This shirt is eco-friendly and natural.', 'Buzzwords': ['eco-friendly', 'natural'], 'Certifications': [], 'Classification': 'Marketing Fluff', 'Reasoning': 'Contains vague buzzwords: eco-friendly, natural without certification.'}
{'Input': 'This cotton is GOTS-certified and produced by a B-Corp brand.', 'Buzzwords': [], 'Certifications': ['GOTS', 'B-Corp'], 'Classification': 'Evidence-Based', 'Reasoning': 'Mentions certification: GOTS, B-Corp'}
{'Input': 'Our packaging is 100% recyclable and FSC-certified.', 'Buzzwords': [], 'Certifications': ['FSC'], 'Classification': 'Evidence-Based', 'Reasoning': 'Mentions certification: FSC'}
{'Input': 'Made with sustainable materials for a greener future.', 'Buzzwords': ['green', 'sustainable'], 'Certifications': [], 'Classification': 'Marketing Fluff', 'Reasoning': 'Contains vague buzzwords: green, sustainable without certification.'}


In [8]:
def audit_interface(text):
    result = audit_product(text)
    return (
        f"Classification: {result['Classification']}\n"
        f"Buzzwords: {result['Buzzwords']}\n"
        f"Certifications: {result['Certifications']}\n"
        f"Reasoning: {result['Reasoning']}"
    )

demo = gr.Interface(
    fn=audit_interface,
    inputs="text",
    outputs="text",
    title="🌱 Green-Truth Auditor",
    description="Classify product claims as Marketing Fluff or Evidence-Based"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cac23c83af97f033f2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [9]:
!git clone https://github.com/username/repo.git

Cloning into 'repo'...
fatal: could not read Username for 'https://github.com': No such device or address
